In [34]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [ ]:
from config import EMBEDDING_MODEL, VECTORSTORE_DIR

In [36]:
from dotenv import load_dotenv
import os
load_dotenv()
chat_history=[]

In [37]:
embedding = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

In [5]:
vector_store = Chroma(
    persist_directory=VECTORSTORE_DIR,
    embedding_function=embedding
)

C:\Users\Catom\AppData\Local\Temp\ipykernel_1316\77805579.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [38]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

In [39]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=os.getenv("GROQ_AI_KEY")
)


In [41]:
from langchain_core.prompts import PromptTemplate

In [42]:
rag_prompt = PromptTemplate(input_variables=["context", "question"], 
                            template="""
You are a helpful assistant.
Answer ONLY using the context below.
If the answer is not present in the context, say "I don't know".

Context:
{context}

Question:
{question}
""")

In [43]:
def ask_question(question: str):
    docs = retriever.invoke(question)

    print("\n--- RETRIEVED DOCUMENTS ---\n")
    for i, doc in enumerate(docs, start=1):
        print(f"[Chunk {i}]")
        print(doc.page_content)
        print("Metadata:", doc.metadata)
        print("-" * 40)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = rag_prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    sources = set()
    for doc in docs:
        if "source" in doc.metadata:
            sources.add(doc.metadata["source"])

    return response.content, list(sources)


In [44]:
if __name__ == "__main__":
    answer, sources = ask_question("fill the blanks The Floating ____ ?")

    print("\n--- FINAL ANSWER ---\n")
    print(answer)

    print("\n--- SOURCES ---\n")
    for source in sources:
        print(source)



--- RETRIEVED DOCUMENTS ---

[Chunk 1]
The Mysterious Affair at Styles
The Secret Adversary
The Murder on the Links
The Man in the Brown Suit
The Secret of Chimneys
The Murder of Roger Ackroyd
The Big Four
The Mystery of the Blue Train
The Seven Dials Mystery
The Murder at the Vicarage
Giant's Bread
The Floating Admiral
The Sittaford Mystery
Peril at End House
Lord Edgware Dies
Murder on the Orient Express
Unfinished Portrait
Why Didn't They Ask Evans?
Three Act Tragedy
Death in the Clouds
Metadata: {'source': 'data/agatha.txt'}
----------------------------------------

--- FINAL ANSWER ---

The Floating Admiral

--- SOURCES ---

data/agatha.txt


In [45]:
rag_prompt = PromptTemplate(
    input_variables=["chat_history", "context", "question"],
    template="""
You are a helpful assistant.
Use ONLY the context below to answer.
If the answer is not in the context, say "I don't know".

Conversation so far:
{chat_history}

Context:
{context}

Question:
{question}
"""
)

In [46]:
def ask_question(question: str):
    global chat_history

    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    history_text = "\n".join(
        f"User: {h['user']}\nAssistant: {h['assistant']}"
        for h in chat_history[-3:]
    )

    prompt = rag_prompt.format(
        chat_history=history_text,
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    chat_history.append({
        "user": question,
        "assistant": response.content
    })

    sources = {doc.metadata.get("source", "") for doc in docs}

    return response.content, list(sources)
